![DepthDif banner](https://raw.githubusercontent.com/simon-donike/DepthDif/main/docs/assets/branding/banner_depthdif.png)

# 🌊 DepthDif ocean-temperature explorer

Welcome! This notebook turns scattered ARGO ocean measurements into a smooth temperature map for the place and week you choose. You can optionally add OSTIA sea-surface temperature for extra surface context, or skip it and let the model work from ARGO alone. 🧭

The notebook takes care of the behind-the-scenes pieces for you: it fetches the public DepthDif model, prepares the land mask, downloads the matching ARGO files, and then paints the result back onto an interactive map. If you choose OSTIA, you will need a Copernicus Marine login; if you press **Skip OSTIA**, no Copernicus login is needed. ✨


## ⚙️ 1. Get the notebook ready
This first step installs the tools the notebook needs and prepares the interactive map widgets. It may take a minute, but you only need to run it once per Colab session. ☕

In [1]:
!pip install -q --upgrade "numpy<2.1" "click>=8.2.1" depth-recon copernicusmarine ipywidgets ipyleaflet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# @title 1.1. Define notebook helpers { display-mode: "form" }
from __future__ import annotations

import base64
from datetime import date
from io import BytesIO
import json
import os
from pathlib import Path

from IPython.display import HTML, clear_output, display
from ipyleaflet import DrawControl, GeoJSON, ImageOverlay, LayersControl, Map, Popup, basemaps
import ipywidgets as widgets
from matplotlib.colors import LinearSegmentedColormap, Normalize
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform as transform_coordinates, transform_bounds
import yaml

import depth_recon.inference.api as depthdif_api

try:
    from google.colab import output

    # Colab needs this for third-party ipywidgets such as ipyleaflet.
    output.enable_custom_widget_manager()
except Exception:
    pass

DEPTH_LEVEL_OPTIONS = (
    (0, 'surface', 'Surface (0 m)'),
    (10, '10m', '10 m'),
    (50, '50m', '50 m'),
    (100, '100m', '100 m'),
    (250, '250m', '250 m'),
    (500, '500m', '500 m'),
    (1000, '1000m', '1000 m'),
    (2000, '2000m', '2000 m'),
    (2500, '2500m', '2500 m'),
)
TEMPERATURE_RANGE_C = (0.0, 30.0)
TEMPERATURE_CMAP = LinearSegmentedColormap.from_list(
    'depthdif_temperature', ['#1d4ed8', '#facc15', '#dc2626']
)
STATE = {
    'hf_repo_id': 'simon-donike/DepthDif',
    'hf_revision': 'main',
    'cache_dir': Path('/content/depthdif_cache'),
    'output_root': Path('/content/depthdif_outputs'),
    'device': 'auto',
    'batch_size': 32,
    'skip_ostia': False,
    'copernicus_username': os.environ.get('COPERNICUSMARINE_SERVICE_USERNAME', ''),
    'copernicus_token': os.environ.get('COPERNICUSMARINE_SERVICE_PASSWORD', ''),
    'year': 2018,
    'iso_week': 25,
    'rectangle': None,
    'public_assets': None,
    'run_dir': None,
}


def _html_status(message: str, kind: str = 'info') -> HTML:
    """Return a compact notebook status message."""
    colors = {'info': '#334155', 'ok': '#166534', 'warn': '#92400e'}
    color = colors.get(kind, colors['info'])
    return HTML(f'<span style="color:{color}">{message}</span>')


def _next_cell_html(message: str = 'Continue in the next cell') -> str:
    """Return friendly HTML that points users to the next notebook cell."""
    return f'<br><span style="color:#166534;font-weight:600">👇 {message}</span>'


def _ostia_quality_warning_html() -> str:
    """Return a prominent warning for ARGO-only inference."""
    return (
        '<br><div style="border:3px solid #f97316;background:#fff7ed;color:#7c2d12;padding:14px 16px;margin:8px 0;font-size:18px;font-weight:800;line-height:1.35;">'
        'Warning: OSTIA is skipped. Output quality will be severely degraded because the model has no sea-surface temperature context.'
        '</div>'
    )


def _set_widgets_disabled(items: list[widgets.Widget], disabled: bool) -> None:
    """Toggle widgets that support a disabled state."""
    for item in items:
        if hasattr(item, 'disabled'):
            item.disabled = disabled


def iso_weeks_in_year(year: int) -> list[int]:
    """Return the valid ISO week numbers for one calendar year."""
    return list(range(1, date(int(year), 12, 28).isocalendar().week + 1))


def _runtime_widgets(reset: bool = False) -> dict[str, widgets.Widget]:
    """Create or return the runtime configuration widgets."""
    if reset or 'runtime_widgets' not in STATE:
        STATE['runtime_widgets'] = {
            'cache_dir': widgets.Text(
                value=str(STATE['cache_dir']), description='Cache', layout=widgets.Layout(width='520px')
            ),
            'output_root': widgets.Text(
                value=str(STATE['output_root']), description='Outputs', layout=widgets.Layout(width='520px')
            ),
            'device': widgets.Dropdown(options=['auto', 'cuda', 'cpu'], value=str(STATE['device']), description='Device'),
            'batch_size': widgets.BoundedIntText(value=int(STATE['batch_size']), min=1, max=128, description='Batch size'),
            'download_button': widgets.Button(description='OK', icon='check', button_style='primary'),
            'status': widgets.HTML(''),
        }
    return STATE['runtime_widgets']


def _sync_runtime_settings() -> None:
    """Copy current runtime widget values into STATE."""
    runtime = _runtime_widgets()
    STATE['cache_dir'] = Path(runtime['cache_dir'].value).expanduser()
    STATE['output_root'] = Path(runtime['output_root'].value).expanduser()
    STATE['device'] = runtime['device'].value
    STATE['batch_size'] = int(runtime['batch_size'].value)


def configure_depthdif_runtime() -> None:
    """Display runtime options that are safe for most Colab users to change."""
    runtime = _runtime_widgets()
    display(widgets.VBox([
        widgets.HTML(
            f'Hugging Face release: <b>{STATE["hf_repo_id"]}</b> @ <b>{STATE["hf_revision"]}</b>'
        ),
        runtime['cache_dir'],
        runtime['output_root'],
        widgets.HBox([runtime['device'], runtime['batch_size']]),
        widgets.HBox([runtime['download_button'], runtime['status']]),
    ]))


def _auth_widgets(reset: bool = False) -> dict[str, widgets.Widget]:
    """Create or return Copernicus Marine credential widgets."""
    if reset or 'auth_widgets' not in STATE:
        STATE['auth_widgets'] = {
            'username': widgets.Text(
                value=str(STATE.get('copernicus_username', '')),
                description='Username',
                layout=widgets.Layout(width='420px'),
            ),
            'token': widgets.Password(
                value=str(STATE.get('copernicus_token', '')),
                description='Token/key',
                layout=widgets.Layout(width='420px'),
            ),
            'button': widgets.Button(description='OK', icon='check', button_style='primary'),
            'skip_button': widgets.Button(description='Skip OSTIA', icon='step-forward', button_style='warning'),
            'status': widgets.HTML(''),
        }
    return STATE['auth_widgets']


def _sync_copernicus_environment(required: bool = False) -> tuple[str, str]:
    """Store Copernicus credentials in environment variables for the current runtime."""
    auth = _auth_widgets()
    username = auth['username'].value.strip()
    token = auth['token'].value
    if required and (not username or not token):
        raise ValueError('Please enter your Copernicus Marine username and token/key, or press Skip OSTIA to continue with ARGO only.')
    STATE['copernicus_username'] = username
    STATE['copernicus_token'] = token
    if username:
        os.environ['COPERNICUSMARINE_SERVICE_USERNAME'] = username
    if token:
        # The toolbox reads the API key/token from its password environment variable.
        os.environ['COPERNICUSMARINE_SERVICE_PASSWORD'] = token
    return username, token


def authenticate_copernicusmarine() -> None:
    """Open Copernicus Marine username and token inputs for OSTIA downloads."""
    auth = _auth_widgets(reset=True)

    def handle_click(_button: widgets.Button) -> None:
        try:
            _sync_copernicus_environment(required=True)
        except ValueError as exc:
            auth['status'].value = f'<span style="color:#92400e">{exc}</span>'
            return
        STATE['skip_ostia'] = False
        _set_widgets_disabled([auth['username'], auth['token'], auth['button'], auth['skip_button']], True)
        auth['status'].value = '<span style="color:#166534">✅ OSTIA settings saved. OSTIA will add sea-surface context to this run.</span>' + _next_cell_html()

    def handle_skip(_button: widgets.Button) -> None:
        STATE['skip_ostia'] = True
        STATE['copernicus_username'] = ''
        STATE['copernicus_token'] = ''
        _set_widgets_disabled([auth['username'], auth['token'], auth['button'], auth['skip_button']], True)
        auth['status'].value = _ostia_quality_warning_html() + '<span style="color:#92400e">⏭️ OSTIA skipped. The run will use ARGO observations only.</span>' + _next_cell_html()

    auth['button'].on_click(handle_click)
    auth['skip_button'].on_click(handle_skip)
    display(widgets.VBox([
        widgets.HTML('🌡️ OSTIA adds a sea-surface temperature layer. Load your Copernicus Marine credentials to include it, or choose Skip OSTIA for a simpler ARGO-only run.'),
        auth['username'],
        auth['token'],
        widgets.HBox([auth['button'], auth['skip_button'], auth['status']]),
    ]))


def rectangle_from_geojson(geo_json: dict) -> tuple[float, float, float, float]:
    """Extract lon_min, lat_min, lon_max, lat_max from a drawn rectangle."""
    geometry = geo_json.get('geometry', geo_json)
    if geometry.get('type') == 'FeatureCollection':
        geometry = geometry['features'][0]['geometry']
    coords = geometry['coordinates'][0]
    lons = [float(point[0]) for point in coords]
    lats = [float(point[1]) for point in coords]
    return min(lons), min(lats), max(lons), max(lats)


def rectangle_area_km2(bounds: tuple[float, float, float, float]) -> float:
    """Estimate a lat/lon rectangle area in square kilometers."""
    lon_min, lat_min, lon_max, lat_max = bounds
    earth_radius_km = 6371.0088
    # Spherical rectangle area keeps high-latitude boxes from being overestimated.
    lon_width_rad = abs(np.radians(lon_max - lon_min))
    lat_band = abs(np.sin(np.radians(lat_max)) - np.sin(np.radians(lat_min)))
    return float((earth_radius_km ** 2) * lon_width_rad * lat_band)


def select_week_and_bbox() -> None:
    """Display ISO week controls and a Leaflet rectangle picker."""
    year_options = list(range(2010, 2026))
    default_year = int(STATE.get('year', 2018))
    default_week = int(STATE.get('iso_week', 25))
    year_widget = widgets.Dropdown(options=year_options, value=default_year, description='Year')
    week_options = iso_weeks_in_year(default_year)
    week_widget = widgets.Dropdown(
        options=week_options,
        value=default_week if default_week in week_options else week_options[-1],
        description='ISO week',
    )
    rectangle_status = widgets.HTML('🖊️ Draw one rectangle on the map to choose your ocean area.')
    continue_button = widgets.Button(description='OK', icon='check', button_style='primary')
    edit_button = widgets.Button(description='Edit area', icon='pencil')
    edit_button.layout.display = 'none'
    draw_map = Map(
        center=(20.0, 0.0),
        zoom=2,
        basemap=basemaps.Esri.WorldImagery,
        scroll_wheel_zoom=True,
    )
    draw_control = DrawControl(
        rectangle={'shapeOptions': {'color': '#0f766e', 'fillOpacity': 0.12}},
        polygon={},
        polyline={},
        circle={},
        circlemarker={},
        marker={},
    )

    def sync_week_options(change: dict) -> None:
        """Keep the ISO-week dropdown valid when the year changes."""
        weeks = iso_weeks_in_year(int(change['new']))
        current_week = int(week_widget.value)
        week_widget.options = weeks
        week_widget.value = current_week if current_week in weeks else weeks[-1]

    def handle_draw(target: DrawControl, action: str, geo_json: dict) -> None:
        """Store the latest rectangle drawn in the Leaflet widget."""
        if action in {'created', 'edited'}:
            bounds = rectangle_from_geojson(geo_json)
            area_km2 = rectangle_area_km2(bounds)
            STATE['rectangle'] = bounds
            STATE['rectangle_area_km2'] = area_km2
            rectangle_status.value = (
                '✅ Area selected: '
                f'lon {bounds[0]:.3f} to {bounds[2]:.3f}, '
                f'lat {bounds[1]:.3f} to {bounds[3]:.3f}; '
                f'about {area_km2:,.0f} km²'
            )
        elif action == 'deleted':
            STATE['rectangle'] = None
            STATE['rectangle_area_km2'] = None
            rectangle_status.value = '🖊️ Draw one rectangle on the map to choose your ocean area.'
            map_box.layout.display = ''
            continue_button.layout.display = ''
            edit_button.layout.display = 'none'

    def handle_continue(_button: widgets.Button) -> None:
        """Confirm the selected week and rectangle before inference."""
        if STATE.get('rectangle') is None:
            rectangle_status.value = '<span style="color:#92400e">🖊️ Please draw one rectangle on the map before continuing.</span>'
            return
        STATE['year'] = int(year_widget.value)
        STATE['iso_week'] = int(week_widget.value)
        bounds = STATE['rectangle']
        area_km2 = float(STATE.get('rectangle_area_km2') or rectangle_area_km2(bounds))
        STATE['rectangle_area_km2'] = area_km2
        _set_widgets_disabled([year_widget, week_widget, continue_button, edit_button], True)
        rectangle_status.value = (
            '<span style="color:#166534">'
            f'✅ Selection saved for {STATE["year"]} ISO week {STATE["iso_week"]}.<br>'
            f'Bounding box: lon {bounds[0]:.3f} to {bounds[2]:.3f}, '
            f'lat {bounds[1]:.3f} to {bounds[3]:.3f}.<br>'
            f'Approximate area: {area_km2:,.0f} km².</span>'
        ) + _next_cell_html()
        map_box.layout.display = 'none'
        continue_button.layout.display = 'none'
        edit_button.layout.display = 'none'

    def handle_edit(_button: widgets.Button) -> None:
        """Reopen the map so the selected rectangle can be changed."""
        map_box.layout.display = ''
        continue_button.layout.display = ''
        edit_button.layout.display = 'none'
        rectangle_status.value = '<span style="color:#334155">🖊️ The map is open again. Adjust the rectangle, then press Continue.</span>'

    year_widget.observe(sync_week_options, names='value')
    draw_control.on_draw(handle_draw)
    continue_button.on_click(handle_continue)
    edit_button.on_click(handle_edit)
    draw_map.add_control(draw_control)
    map_box = widgets.VBox([draw_map])
    STATE['year_widget'] = year_widget
    STATE['week_widget'] = week_widget
    display(widgets.HBox([year_widget, week_widget, continue_button, edit_button]), rectangle_status, map_box)


def resolve_depthdif_assets() -> None:
    """Show the runtime controls and download public assets after confirmation."""
    if not hasattr(depthdif_api, 'resolve_public_inference_assets'):
        raise RuntimeError('The installed depth-recon package is too old. Please rerun the install cell with --upgrade, then restart the runtime if needed.')
    runtime = _runtime_widgets(reset=True)
    configure_depthdif_runtime()
    runtime['status'].value = '<span style="color:#334155">Choose paths, device, and batch size, then press <b>OK</b>.</span>'

    def handle_download(_button: widgets.Button) -> None:
        """Resolve the public model, config, and land-mask files from Hugging Face."""
        _sync_runtime_settings()
        _set_widgets_disabled(
            [runtime['cache_dir'], runtime['output_root'], runtime['device'], runtime['batch_size'], runtime['download_button']],
            True,
        )
        runtime['status'].value = '<span style="color:#334155">Settings saved. Preparing downloads...</span>'
        total_files = 5
        progress_status = widgets.HTML(f'<b>Files ready: 0/{total_files}</b>')
        file_status = widgets.HTML('')
        completed: set[str] = set()

        def report_progress(event: str, name: str, path: Path) -> None:
            """Reflect package asset-resolution progress in notebook widgets."""
            labels = {
                'cached': '✅ Reusing cached',
                'downloading': '⬇️ Downloading',
                'downloaded': '✅ Downloaded',
                'builtin': '✅ Using built-in fallback for',
                'packaged': '✅ Using packaged',
            }
            file_status.value = f'{labels.get(event, event)} {name}'
            if event in {'cached', 'downloaded', 'builtin', 'packaged'}:
                completed.add(name)
                progress_status.value = f'<b>Files ready: {min(total_files, len(completed))}/{total_files}</b>'

        display(widgets.VBox([progress_status, file_status]))
        try:
            STATE['public_assets'] = depthdif_api.resolve_public_inference_assets(
                config_repo=STATE['hf_repo_id'],
                revision=STATE['hf_revision'],
                cache_dir=STATE['cache_dir'],
                progress_callback=report_progress,
            )
            progress_status.value = f'<b>Files ready: {total_files}/{total_files}</b>'
            bundle = STATE['public_assets']
            assets = bundle.assets
            file_status.value = '✅ All public DepthDif files are ready.'
            display(HTML(
                '<br>'.join([
                    f'Model config: {assets.model_config}',
                    f'Data config: {assets.data_config}',
                    f'Train config: {assets.train_config}',
                    f'Checkpoint: {assets.checkpoint}',
                    f'Land mask: {bundle.land_mask_path}',
                ])
            ))
            runtime['status'].value = '<span style="color:#166534">✅ Settings saved and downloads ready.</span>'
            display(_html_status('👇 Continue in the next cell.', 'ok'))
        except Exception as exc:
            _set_widgets_disabled(
                [runtime['cache_dir'], runtime['output_root'], runtime['device'], runtime['batch_size'], runtime['download_button']],
                False,
            )
            runtime['status'].value = f'<span style="color:#92400e">Download failed: {exc}</span>'
            raise

    runtime['download_button'].on_click(handle_download)


def prediction_tif_from_run(run_dir: Path, suffix: str = 'surface') -> Path:
    """Resolve one exported prediction GeoTIFF from a DepthDif run directory."""
    run_dir = Path(run_dir)
    summary_path = run_dir / 'run_summary.yaml'
    if summary_path.exists():
        summary = yaml.safe_load(summary_path.read_text()) or {}
        for record in summary.get('depth_exports', []):
            label = str(record.get('label', '')).lower()
            record_suffix = str(record.get('suffix', '')).lower()
            if suffix.lower() in {label, record_suffix}:
                path = Path(record['prediction_tif_path'])
                return path if path.is_absolute() else run_dir / path
        if summary.get('prediction_tif_path'):
            path = Path(summary['prediction_tif_path'])
            return path if path.is_absolute() else run_dir / path
    matches = sorted(run_dir.glob(f'*_prediction_{suffix}.tif'))
    if not matches:
        matches = sorted(run_dir.glob('*_prediction*.tif'))
    if not matches:
        raise FileNotFoundError(f'No prediction map was found in {run_dir}. Please run inference first.')
    return matches[0]


def depth_level_metadata(run_dir: Path, depth_m: int) -> tuple[str, str]:
    """Return the exported raster suffix and display label for a requested depth."""
    run_dir = Path(run_dir)
    summary_path = run_dir / 'run_summary.yaml'
    if summary_path.exists():
        summary = yaml.safe_load(summary_path.read_text()) or {}
        for record in summary.get('depth_exports', []):
            requested_depth = float(record.get('requested_depth_m', -1.0))
            if abs(requested_depth - float(depth_m)) < 0.5:
                suffix = str(record.get('suffix', '')).strip()
                label = str(record.get('label', suffix)).strip() or suffix
                if suffix:
                    return suffix, label
    for option_depth, suffix, label in DEPTH_LEVEL_OPTIONS:
        if int(option_depth) == int(depth_m):
            return suffix, label
    raise ValueError(f'Unsupported depth level: {depth_m} m')


def geotiff_leaflet_payload(
    tif_path: Path,
    *,
    percentile_clip: tuple[float, float] = (2.0, 98.0),
) -> tuple[str, tuple[tuple[float, float], tuple[float, float]], tuple[float, float], tuple[float, float]]:
    """Convert one GeoTIFF to a Leaflet-ready PNG, bounds, and display range."""
    with rasterio.open(tif_path) as src:
        source_crs = src.crs or 'EPSG:4326'
        # Leaflet/ipyleaflet maps default to Web Mercator. Reproject only the
        # visualization PNG to EPSG:3857, then pass Leaflet WGS84 corner bounds.
        with WarpedVRT(src, crs='EPSG:3857', src_crs=source_crs, resampling=Resampling.nearest) as vrt:
            data = vrt.read(1, masked=True).astype('float32')
            west, south, east, north = transform_bounds(vrt.crs, 'EPSG:4326', *vrt.bounds, densify_pts=21)

    values = np.ma.filled(data, np.nan)
    valid = np.isfinite(values) & ~np.isclose(values, 0.0)
    if not valid.any():
        raise ValueError(f'Prediction raster has no finite values: {tif_path}')
    valid_values = values[valid]
    raw_range = (float(np.nanmin(valid_values)), float(np.nanmax(valid_values)))
    low, high = [float(value) for value in np.nanpercentile(valid_values, percentile_clip)]
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low, high = raw_range
    if high <= low:
        high = low + 1.0
    clipped = np.clip(values, low, high)
    normalized = np.clip((clipped - low) / (high - low), 0.0, 1.0)

    # Leaflet receives an RGBA PNG because browsers cannot render GeoTIFFs directly.
    rgba = TEMPERATURE_CMAP(normalized)
    rgba[..., 3] = np.where(valid, 1.0, 0.0)
    buffer = BytesIO()
    plt.imsave(buffer, rgba, format='png')
    data_uri = 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('ascii')
    return data_uri, ((south, west), (north, east)), (low, high), raw_range


def temperature_colorbar_payload(value_range_c: tuple[float, float]) -> str:
    """Return a compact color-bar PNG data URI for the current map scale."""
    fig, ax = plt.subplots(figsize=(5.8, 0.7))
    fig.subplots_adjust(bottom=0.45, top=0.82, left=0.08, right=0.98)
    norm = Normalize(vmin=float(value_range_c[0]), vmax=float(value_range_c[1]))
    colorbar = fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=TEMPERATURE_CMAP),
        cax=ax,
        orientation='horizontal',
    )
    colorbar.set_label('Temperature (°C)')
    buffer = BytesIO()
    fig.savefig(buffer, format='png', dpi=140, transparent=True)
    plt.close(fig)
    return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('ascii')


def _declared_geojson_crs(data: dict) -> str | None:
    """Return a legacy GeoJSON CRS name when one is present."""
    crs = data.get('crs')
    if not isinstance(crs, dict):
        return None
    properties = crs.get('properties')
    if not isinstance(properties, dict):
        return None
    name = properties.get('name')
    return str(name) if name else None


def _transform_geojson_coordinates(coords: object, src_crs: str) -> object:
    """Transform GeoJSON coordinates to WGS84 lon/lat for Leaflet display."""
    if (
        isinstance(coords, (list, tuple))
        and len(coords) >= 2
        and isinstance(coords[0], (int, float))
        and isinstance(coords[1], (int, float))
    ):
        xs, ys = transform_coordinates(src_crs, 'EPSG:4326', [float(coords[0])], [float(coords[1])])
        tail = list(coords[2:]) if len(coords) > 2 else []
        return [float(xs[0]), float(ys[0]), *tail]
    if isinstance(coords, list):
        return [_transform_geojson_coordinates(item, src_crs) for item in coords]
    return coords


def geojson_for_leaflet(path: Path) -> dict:
    """Load GeoJSON and convert declared non-WGS84 coordinates for Leaflet."""
    data = json.loads(Path(path).read_text())
    src_crs = _declared_geojson_crs(data)
    if not src_crs or src_crs.upper().endswith('4326'):
        return data

    def transform_geometry(geometry: dict) -> None:
        """Transform one GeoJSON geometry in place."""
        if geometry.get('type') == 'GeometryCollection':
            for item in geometry.get('geometries', []):
                transform_geometry(item)
            return
        if 'coordinates' in geometry:
            geometry['coordinates'] = _transform_geojson_coordinates(geometry['coordinates'], src_crs)

    features = data.get('features', [])
    for feature in features:
        geometry = feature.get('geometry') if isinstance(feature, dict) else None
        if isinstance(geometry, dict):
            transform_geometry(geometry)
    data.pop('crs', None)
    return data


def add_geojson_layer(map_obj: Map, path: Path, name: str) -> None:
    """Add a WGS84 GeoJSON layer to a Leaflet map when the file exists."""
    path = Path(path)
    if path.exists():
        map_obj.add_layer(GeoJSON(data=geojson_for_leaflet(path), name=name))


def _format_argo_popup(feature: dict) -> tuple[tuple[float, float], str]:
    """Build popup location and HTML for one ARGO GeoJSON point feature."""
    coords = feature.get('geometry', {}).get('coordinates', [np.nan, np.nan])
    lon = float(coords[0])
    lat = float(coords[1])
    properties = feature.get('properties', {})
    date_value = str(properties.get('date', 'unknown'))
    if len(date_value) == 8 and date_value.isdigit():
        date_value = f'{date_value[:4]}-{date_value[4:6]}-{date_value[6:]}'
    html = (
        '<b>ARGO observation</b><br>'
        f'Date: {date_value}<br>'
        f'Lat: {lat:.5f}<br>'
        f'Lon: {lon:.5f}<br>'
        f'Patch: {properties.get("patch_id", "")}'
    )
    return (lat, lon), html


def add_argo_points_layer(map_obj: Map, path: Path, info_widget: widgets.HTML) -> None:
    """Add clickable ARGO points with coordinate/date popup text."""
    path = Path(path)
    if not path.exists():
        return
    data = geojson_for_leaflet(path)
    layer = GeoJSON(data=data, name='ARGO observations')
    popup_state = {'layer': None}

    def handle_click(event: dict | None = None, feature: dict | None = None, **kwargs: object) -> None:
        """Show location and date for the clicked ARGO point."""
        feature = feature or kwargs.get('feature')
        if not isinstance(feature, dict):
            return
        location, html = _format_argo_popup(feature)
        info_widget.value = html
        old_popup = popup_state.get('layer')
        if old_popup is not None:
            try:
                map_obj.remove_layer(old_popup)
            except Exception:
                pass
        popup = Popup(location=location, child=widgets.HTML(value=html), close_button=True, auto_close=True, close_on_escape_key=True)
        popup_state['layer'] = popup
        map_obj.add_layer(popup)

    if hasattr(layer, 'on_click'):
        layer.on_click(handle_click)
    map_obj.add_layer(layer)


def build_result_map(run_dir: Path, rectangle: tuple[float, float, float, float], *, depth_suffix: str = 'surface', depth_label: str = 'Surface (0 m)') -> tuple[Map, widgets.HTML, tuple[float, float], tuple[float, float]]:
    """Build a Leaflet result map with the prediction overlay and ARGO points."""
    tif_path = prediction_tif_from_run(run_dir, suffix=depth_suffix)
    data_uri, bounds, display_range, raw_range = geotiff_leaflet_payload(tif_path)
    (south, west), (north, east) = bounds
    center = ((south + north) / 2.0, (west + east) / 2.0)
    result_map = Map(center=center, zoom=4, basemap=basemaps.Esri.WorldImagery, scroll_wheel_zoom=True)
    result_map.add_layer(ImageOverlay(url=data_uri, bounds=bounds, name=f'DepthDif {depth_label}'))
    argo_info = widgets.HTML('<span style="color:#334155">Click an ARGO point to see its date and coordinates.</span>')
    add_argo_points_layer(result_map, next(iter(sorted(Path(run_dir).glob('*_argo_points.geojson'))), Path('__missing__')), argo_info)
    add_geojson_layer(result_map, next(iter(sorted(Path(run_dir).glob('*_patches.geojson'))), Path('__missing__')), 'Selected patches')
    result_map.add_control(LayersControl(position='topright'))
    return result_map, argo_info, display_range, raw_range


def run_depthdif_inference() -> None:
    """Run public API inference with automatic ARGO and optional OSTIA downloads."""
    if not STATE.get('skip_ostia', False):
        if not STATE.get('copernicus_username') or not STATE.get('copernicus_token'):
            raise ValueError('Please run the OSTIA cell and press OK, or press Skip OSTIA.')
        os.environ['COPERNICUSMARINE_SERVICE_USERNAME'] = STATE['copernicus_username']
        os.environ['COPERNICUSMARINE_SERVICE_PASSWORD'] = STATE['copernicus_token']
    if STATE.get('rectangle') is None:
        raise ValueError('Please draw a rectangle on the map before running inference.')
    if STATE.get('year') is None or STATE.get('iso_week') is None:
        raise ValueError('Please run the week-and-map selection step before inference.')
    if STATE.get('public_assets') is None:
        raise ValueError('Please run the runtime cell and press OK before inference.')

    progress_output = widgets.Output()
    tile_progress = widgets.IntProgress(value=0, min=0, max=1, description='Tiles')
    batch_status = widgets.HTML('')
    display(widgets.VBox([progress_output, tile_progress, batch_status]))

    def show_progress(message: str, kind: str = 'info') -> None:
        """Append one inference progress message to the notebook output."""
        with progress_output:
            display(_html_status(message, kind))

    effective_device = depthdif_api.choose_device(str(STATE['device']))
    if effective_device.type != 'cuda':
        with progress_output:
            display(HTML(
                '<div style="border:3px solid #dc2626;background:#fff7ed;color:#7f1d1d;padding:14px 16px;margin:8px 0;font-size:18px;font-weight:800;line-height:1.35;">'
                '⚠️🐢 WARNING: CUDA is not active. This run will use '
                f'<code>{effective_device}</code> and can take a long time. '
                'Switch Colab to a GPU runtime and choose <code>cuda</code> or <code>auto</code> for a much faster run. 🚨🔥'
                '</div>'
            ))

    def report_inference_progress(event: str, name: str, path: Path) -> None:
        """Render public-inference API progress events in the notebook."""
        path = Path(path)
        messages = {
            'argo_prepare_start': f'📥 Starting ARGO profile download/preparation in: {path}',
            'argo_download_start': f'📥 Starting ARGO archive download: {name}',
            'argo_download_done': f'✅ ARGO archive downloaded: {path}',
            'argo_archive_cached': f'✅ Reusing cached ARGO archive: {path}',
            'argo_unzip_start': f'📦 Unzipping ARGO archive into monthly profile files: {name}',
            'argo_unzip_done': f'✅ ARGO archive unzipped: {name} in {path}',
            'argo_cached': f'✅ ARGO monthly profile files already exist: {path}',
            'argo_ready': f'✅ ARGO profile files ready: {path}',
            'ostia_download_start': f'📥 Starting OSTIA download/preparation in: {path}',
            'ostia_ready': f'✅ OSTIA file ready: {path}',
            'patch_grid_start': '🧭 Selecting ocean patches from the land mask and bounding box...',
            'patch_grid_done': f'✅ Selected inference grid: {name}',
            'profile_alignment_start': '🧪 Building aligned ARGO profiles for the selected patches...',
            'inference_start': f'🚀 Starting inference run: {name}',
            'export_start': '🗺️ Writing GeoTIFF and GeoJSON outputs...',
            'export_done': f'✅ Output files written in: {path}',
        }
        if event == 'inference_start':
            total = int(str(name).split()[0])
            tile_progress.max = max(1, total)
            tile_progress.value = 0
            batch_status.value = f'<span style="color:#334155">Tiles processed: 0/{total}</span>'
        if event == 'profile_batch_start':
            batch_status.value = f'<span style="color:#334155">Building aligned ARGO tensors: {name}</span>'
            return
        if event == 'profile_batch_done':
            batch_status.value = f'<span style="color:#334155">Aligned ARGO tensors ready: {name}</span>'
            return
        if event == 'inference_batch_start':
            batch_status.value = f'<span style="color:#334155">Running model inference: {name}</span>'
            return
        if event == 'inference_batch':
            processed_text, total_text = str(name).split()[0].split('/')
            tile_progress.max = max(tile_progress.max, int(total_text))
            tile_progress.value = min(tile_progress.max, int(processed_text))
            batch_status.value = f'<span style="color:#334155">Tiles processed: {name}</span>'
            return
        if event in messages:
            show_progress(messages[event], 'ok' if event.endswith(('_done', '_ready', '_cached')) else 'info')

    show_progress('📥 Starting ARGO profile download/preparation...')
    batch_size = int(STATE['batch_size'])
    run_dir = depthdif_api.run_week_inference(
        year=int(STATE['year']),
        iso_week=int(STATE['iso_week']),
        rectangle=STATE['rectangle'],
        output_root=STATE['output_root'],
        device=STATE['device'],
        config_repo=STATE['hf_repo_id'],
        revision=STATE['hf_revision'],
        cache_dir=STATE['cache_dir'],
        auto_download_argo=True,
        auto_download_ostia=not bool(STATE.get('skip_ostia', False)),
        copernicus_username=STATE.get('copernicus_username') or None,
        copernicus_token=STATE.get('copernicus_token') or None,
        batch_size=batch_size,
        force_download=False,
        progress_callback=report_inference_progress,
    )
    STATE['run_dir'] = Path(run_dir)
    display(_html_status(f'✅ Finished! Your result files are in: {STATE["run_dir"]}.<br>👇 Continue in the next cell.', 'ok'))


def display_depthdif_result(depth_m: int = 0) -> None:
    """Display the exported DepthDif prediction on a Leaflet map."""
    if STATE.get('run_dir') is None:
        raise ValueError('Run run_depthdif_inference() first.')
    depth_values = [depth for depth, _suffix, _label in DEPTH_LEVEL_OPTIONS]
    selected_depth = int(depth_m) if int(depth_m) in depth_values else 0
    depth_slider = widgets.SelectionSlider(
        options=[(label, depth) for depth, _suffix, label in DEPTH_LEVEL_OPTIONS],
        value=selected_depth,
        description='Depth',
        continuous_update=False,
        layout=widgets.Layout(width='620px'),
    )
    output = widgets.Output()

    def render_depth(change: dict | None = None) -> None:
        """Refresh the map when the selected depth changes."""
        with output:
            clear_output(wait=True)
            suffix, label = depth_level_metadata(STATE['run_dir'], int(depth_slider.value))
            if STATE.get('skip_ostia', False):
                display(HTML(_ostia_quality_warning_html()))
            result_map, argo_info, display_range, raw_range = build_result_map(
                STATE['run_dir'],
                STATE['rectangle'],
                depth_suffix=suffix,
                depth_label=label,
            )
            display(_html_status(
                f'🌡️ Showing {label}. Color stretch uses the 2nd-98th percentiles '
                f'({display_range[0]:.2f} to {display_range[1]:.2f} °C); '
                f'raw nonzero range is {raw_range[0]:.2f} to {raw_range[1]:.2f} °C. '
                'Zero-valued pixels are transparent.',
                'info',
            ))
            display(HTML(f'<img src="{temperature_colorbar_payload(display_range)}" alt="Temperature color scale using prediction percentiles" style="max-width:620px;width:100%;">'))
            display(result_map)
            display(argo_info)

    depth_slider.observe(render_depth, names='value')
    render_depth()
    display(widgets.VBox([depth_slider, output]))
    display(_html_status('✅ Result viewer is ready.<br>👇 Continue below when you are done exploring.', 'ok'))

import depth_recon
print("Depth Recon version:", depth_recon.__version__)


Depth Recon version: 0.0.1a11


## 🛠️ 2. Choose how the run should behave
Pick where temporary files and outputs should live, then choose the compute device and batch size. Press **OK** to save those choices and prepare the public DepthDif files. The default settings are a good starting point: **auto** lets Colab choose the best available device, and batch size **32** is a practical balance for most sessions. Existing matching files are reused automatically, while missing files are downloaded for you. 📦


In [3]:
resolve_depthdif_assets() # choose settings, then press OK

In [4]:
authenticate_copernicusmarine() # press OK to save OSTIA credentials, or skip OSTIA

## 🚀 3. Create your ocean map

### 🗺️ 3.1 Pick a week and draw your area
Choose the year and ISO week you want to explore, then draw a rectangle around your ocean region and press **OK** to save it. Smaller areas finish faster and are great for trying things out; larger areas give broader maps but take longer. As a rough guide, a whole-world run can take about an hour, so start with a focused region if you are experimenting. 🌍


In [5]:
select_week_and_bbox() # draw a rectangle, then press OK

HTML(value='🖊️ Draw one rectangle on the map to choose your ocean area.')

### 🚀 3.2 Let DepthDif do the work
Now the notebook gathers the needed ocean observations, runs the model, and writes the map files. If OSTIA was skipped, the model will rely on ARGO observations only; otherwise it will also use the sea-surface temperature layer. 🧪

In [7]:
run_depthdif_inference() # build the temperature prediction map

Preparing public ARGO inference: selected_date=20180620, iso_week=2018-W25, selected_patches=1, batch_size=32, ostia_conditioning=disabled, rectangle=(-8.833008, 42.972502, -0.615234, 48.107431)


Finalizing depthdif_argo_20180620_prediction_surface.tif: 100%|██████████| 1/1 [00:00<00:00, 183.28chunk/s]
Finalizing depthdif_argo_20180620_prediction_10m.tif: 100%|██████████| 1/1 [00:00<00:00, 216.35chunk/s]
Finalizing depthdif_argo_20180620_prediction_50m.tif: 100%|██████████| 1/1 [00:00<00:00, 205.31chunk/s]
Finalizing depthdif_argo_20180620_prediction_100m.tif: 100%|██████████| 1/1 [00:00<00:00, 189.86chunk/s]
Finalizing depthdif_argo_20180620_prediction_250m.tif: 100%|██████████| 1/1 [00:00<00:00, 199.80chunk/s]
Finalizing depthdif_argo_20180620_prediction_500m.tif: 100%|██████████| 1/1 [00:00<00:00, 194.23chunk/s]
Finalizing depthdif_argo_20180620_prediction_1000m.tif: 100%|██████████| 1/1 [00:00<00:00, 243.54chunk/s]
Finalizing depthdif_argo_20180620_prediction_2000m.tif: 100%|██████████| 1/1 [00:00<00:00, 178.41chunk/s]
Finalizing depthdif_argo_20180620_prediction_2500m.tif: 100%|██████████| 1/1 [00:00<00:00, 164.97chunk/s]
Finalizing depthdif_argo_20180620_prediction_5000m.

## 🧭 4. Explore the result
Open the finished prediction on an interactive map. Use the depth slider to move from the surface down to deeper layers, and read the colors on a fixed 0-30 °C scale: blue is cooler, yellow is mid-range, and red is warmer. You can also zoom, pan, and toggle the ARGO observation points and selected patch layer to see where the model had direct measurements to work from. 🎨

In [8]:
display_depthdif_result()